> 🔑 **교수자용 답안 노트북** — 학생용 파일: `unit3_python.ipynb`

# 3단원 파이썬 확장 실습
**디지털리터러시 36시간 과정 · 실생활에 유용한 AI 활용 · 4차시 가계·소비**

| 파트 | 내용 |
|---|---|
| A | 지출 메모 → CSV 저장 → 분류 → 합계·비율 → AI 답과 대조 |
| B | 요금제 총요금 함수 → 12개월 vs 24개월 비교 |

- **Shift + Enter**로 위에서부터 실행합니다. **✏️** 직접 채우기 · **🤖** AI 활용
- 모든 데이터는 **수업용 가상 데이터**입니다. 실제 카드·계좌 정보는 넣지 않습니다.

In [ ]:
def 확인(이름, 결과, 정답):
    if 결과 is None:
        print(f"⏳ {이름}: 아직 비어 있어요 — ✏️ 셀을 채우고 다시 실행하세요")
    elif 결과 == 정답:
        print(f"✅ {이름}: 정답입니다! ({결과:,})" if isinstance(결과, int) else f"✅ {이름}: 정답입니다!")
    else:
        print(f"❌ {이름}: 결과 {결과} / 기대 {정답}")

print("준비 완료")

---
## A. 지출 메모 분류·합계

### A-1. 지출 메모를 CSV 파일로 저장
**CSV**는 쉼표로 칸을 나눈 표 파일입니다. 구글 시트·엑셀에서도 열 수 있습니다.

In [ ]:
지출_메모 = """날짜,항목,금액
3/2,동네 마트 장보기,54300
3/3,교통카드 충전,30000
3/5,휴대폰 요금,48000
3/8,약국 감기약,6500
3/10,세제·휴지,17800
3/12,친구와 점심,22000
3/15,동네 병원 진료비,5000
3/18,인터넷 요금,22000
3/21,시장 반찬,12000
3/25,손주 생일 선물,35000
"""

with open("지출.csv", "w", encoding="utf-8") as f:
    f.write(지출_메모)

print("지출.csv 저장 완료 — 왼쪽 📁 파일 목록에서 확인할 수 있어요")

### A-2. CSV 파일 읽기
`csv.DictReader`는 한 줄을 `{"날짜": ..., "항목": ..., "금액": ...}` 딕셔너리로 읽어 줍니다. 금액은 **글자**로 읽히므로 `int()`로 바꿉니다.

In [ ]:
import csv

지출들 = []
with open("지출.csv", encoding="utf-8") as f:
    for 줄 in csv.DictReader(f):
        줄["금액"] = int(줄["금액"])
        지출들.append(줄)

for 지출 in 지출들:
    print(f'{지출["날짜"]:>5}  {지출["항목"]:<12} {지출["금액"]:>8,}원')

### A-3. ✏️ 분류 함수 만들기
항목 이름에 **규칙의 낱말이 들어 있으면** 그 분류로 정합니다. 어느 규칙에도 해당하지 않으면 `"기타"`입니다.
힌트: `for 분류, 낱말들 in 분류_규칙.items():` 안에서 `if 낱말 in 항목:`

In [ ]:
분류_규칙 = {
    "식비":     ["마트", "점심", "반찬"],
    "교통":     ["교통카드"],
    "통신":     ["휴대폰", "인터넷"],
    "의료":     ["약국", "병원"],
    "생활용품": ["세제", "휴지"],
}

def 분류하기(항목):
    for 분류, 낱말들 in 분류_규칙.items():
        for 낱말 in 낱말들:
            if 낱말 in 항목:
                return 분류
    return "기타"

for 지출 in 지출들:
    print(f'{지출["항목"]:<12} → {분류하기(지출["항목"])}')

In [ ]:
확인("약국 감기약", 분류하기("약국 감기약"), "의료")
확인("인터넷 요금", 분류하기("인터넷 요금"), "통신")
확인("손주 생일 선물", 분류하기("손주 생일 선물"), "기타")

### A-4. ✏️ 분류별 합계
딕셔너리에 **누적**합니다: `합계[분류] = 합계.get(분류, 0) + 금액`

In [ ]:
분류별_합계 = {}
for 지출 in 지출들:
    분류 = 분류하기(지출["항목"])
    분류별_합계[분류] = 분류별_합계.get(분류, 0) + 지출["금액"]

전체_합계 = sum(분류별_합계.values())
print(분류별_합계)
print(f"전체 합계: {전체_합계:,}원")

In [ ]:
확인("식비 합계", 분류별_합계.get("식비"), 88300)
확인("통신 합계", 분류별_합계.get("통신"), 70000)
확인("전체 합계", 전체_합계 or None, 252600)

### A-5. 비율 막대그래프와 AI 답 대조
글자(■)로 그리는 그래프는 한글 글꼴 설정 없이 어디서나 보입니다.

In [ ]:
print("분류별 지출 (많은 순)\n")
for 분류, 금액 in sorted(분류별_합계.items(), key=lambda x: -x[1]):
    비율 = 금액 / 전체_합계 * 100 if 전체_합계 else 0
    print(f"{분류:<5} {금액:>8,}원 {비율:5.1f}%  " + "■" * round(비율 / 2))

In [ ]:
# ✏️ 실습지 미션 9에서 AI가 말한 전체 합계를 숫자로 적으세요 (쉼표 없이)
AI_합계 = 252600
계산기_합계 = 252600   # 내가 계산기로 구한 값

print(f"AI      : {AI_합계:,}원")
print(f"계산기  : {계산기_합계:,}원")
print(f"파이썬  : {전체_합계:,}원")
print("✅ 세 값이 모두 같습니다" if AI_합계 == 계산기_합계 == 전체_합계 else "⚠️ 서로 다른 값이 있습니다 — 어느 쪽이 틀렸는지 항목별로 찾아보세요")

#### (선택) 진짜 그래프 그리기 — matplotlib
Colab에서 그래프에 한글을 쓰려면 한글 글꼴을 먼저 설치합니다. (처음 한 번, 약 20초)

In [ ]:
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

글꼴 = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(글꼴):
    try:
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], check=False,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except FileNotFoundError:
        pass   # Colab이 아닌 환경
if os.path.exists(글꼴):
    fm.fontManager.addfont(글꼴)
    plt.rcParams["font.family"] = "NanumGothic"
else:
    plt.rcParams["font.family"] = "Malgun Gothic"   # Windows PC에서 실행할 때
plt.rcParams["axes.unicode_minus"] = False

항목들 = sorted(분류별_합계, key=분류별_합계.get, reverse=True)
plt.figure(figsize=(7, 3.5))
plt.bar(항목들, [분류별_합계[k] for k in 항목들], color="#10684A")
plt.title("3월 분류별 지출 (가상 데이터)")
plt.ylabel("원")
plt.tight_layout()
plt.show()

---
## B. 요금제, 기간을 바꾸면 답이 바뀐다

미션 10의 **가상 요금제**입니다. (실제 통신사 상품이 아닙니다)

| 요금제 | 조건 |
|---|---|
| 가 | 월 33,000원 |
| 나 | 처음 6개월 월 22,000원, 7개월째부터 39,000원 |
| 다 | 월 45,000원 |

### B-1. ✏️ 총요금 함수
`개월수` 동안 낼 요금을 한 달씩 반복하며 더합니다. **할인개월 이하**면 처음요금, 넘으면 이후요금.

In [ ]:
요금제들 = [
    {"이름": "가", "처음요금": 33000, "할인개월": 0, "이후요금": 33000},
    {"이름": "나", "처음요금": 22000, "할인개월": 6, "이후요금": 39000},
    {"이름": "다", "처음요금": 45000, "할인개월": 0, "이후요금": 45000},
]

def 총요금(요금제, 개월수):
    합 = 0
    for 달 in range(1, 개월수 + 1):
        if 달 <= 요금제["할인개월"]:
            합 += 요금제["처음요금"]
        else:
            합 += 요금제["이후요금"]
    return 합

for 요금제 in 요금제들:
    print(요금제["이름"], f"{총요금(요금제, 12):,}원")

In [ ]:
확인("가 12개월", 총요금(요금제들[0], 12) or None, 396000)
확인("나 12개월", 총요금(요금제들[1], 12) or None, 366000)
확인("다 12개월", 총요금(요금제들[2], 12) or None, 540000)

### B-2. 기간별로 비교하면?
1개월부터 24개월까지 누적 요금을 표로 뽑고, **가장 싼 요금제가 바뀌는 달**을 찾습니다.

In [ ]:
print(" 개월 |        가 |        나 |        다 | 가장 쌈")
print("-" * 52)
이전_최저 = None
for 개월 in range(1, 25):
    요금 = {p["이름"]: 총요금(p, 개월) for p in 요금제들}
    최저값 = min(요금.values())
    최저들 = [이름 for 이름, 값 in 요금.items() if 값 == 최저값]
    if len(최저들) > 1:
        최저, 표시 = "=".join(최저들), "  (같음)"
    else:
        최저 = 최저들[0]
        표시 = "  ← 바뀜!" if 이전_최저 and 최저 != 이전_최저 else ""
        이전_최저 = 최저
    if 개월 in (1, 6, 12, 24) or 표시:
        print(f"{개월:>5} | {요금['가']:>9,} | {요금['나']:>9,} | {요금['다']:>9,} | {최저}{표시}")

#### 💬 생각해 보기
- 12개월 기준으로는 **나**, 24개월 기준으로는 **가**가 쌉니다. 17개월째는 두 요금제가 같습니다.
- AI에게 "어떤 요금제가 제일 싸?"라고만 물으면 **몇 개월 기준**으로 답할까요?

🤖 **AI에게 다시 묻기**
```
나 ▶ 위 요금제 3개를 1년 기준과 2년 기준으로 각각 비교해 줘. 어느 달부터 순위가 바뀌는지도 알려 줘.
```
AI 답과 위 표를 대조해 보세요. 👉 **조건을 분명히 줄수록 답이 정확해지고, 계산은 직접 검증합니다.**